# Automated Malaria Cell Detection — Phase 3: testing our hypotheses (A1–A3)
## Does the 97–98 % hold on patients the model has never seen, and does YUV stain normalisation help?

**Team:** J Amathziah (230139): Kaggle runs, repo · C Murali Madhav (230115): code (notebook, scripts) · Ravi Yadav (230131): slides · Pugazhendhi J (230066): report
**Base paper:** G. Marques, A. Ferreras, I. de la Torre-Díez, "An ensemble-based approach for automated medical diagnosis of malaria using EfficientNet," *Multimedia Tools and Applications*, 81, 28061–28078, 2022.
**Baseline (A0):** `Phase2_A0_10fold_PaperExact.ipynb`, the paper's own recipe on all 10 folds, within 1 pp of the paper on every metric.

### The hypotheses, as we stated them before running anything (Phase 2 deck, slides 28–29)
* **H1 · validation.** Split by patient instead of by image. *Expect:* accuracy and MCC fall below the image-split baseline, and fold-to-fold spread widens. *Because:* cells from one patient share stain, lighting and the patient's blood, so a random image split lets the model recognise the patient instead of the parasite.
* **H2 · preprocessing.** RGB → YUV → histogram-equalise Y → RGB before EfficientNet-B0 (Umer *et al.* 2020), patient split kept. *Expect:* higher recall and MCC than H1 on unseen patients; little change on the random split.
* **Falsifiable:** if H2's gain over H1 is smaller than the fold-to-fold SD, we reject H2. If grouped and random splits score the same, H1 is rejected.

| ID | Split | Preprocessing | What it answers |
|---|---|---|---|
| A0 | image (paper) | none | baseline (done, Phase 2) |
| A1 | **patient** | none | H1: how much does patient leakage inflate the scores (A0 vs A1)? |
| A2 | **patient** | **YUV + hist-eq** | H2: does stain normalisation help on unseen patients (A2 vs A1)? |
| A3 | image (paper) | **YUV + hist-eq** | control: does YUV change anything when patients are shared (A3 vs A0)? |

**One change at a time.** Everything else is A0's recipe, unchanged: noisy-student EfficientNet-B0 with the paper's dense head, label smoothing 0.1, Adam 1e-4, ReduceLROnPlateau, 33 epochs × batch 16, best-val_loss checkpoint, the paper's albumentations pipeline, the same seeds per fold, the same 10-model averaging ensemble. A3 even reuses A0's exact split (same fingerprint), so A3 vs A0 compares the same cells, fold by fold; A2 vs A1 likewise share their patient split.

**Decided before the runs** (full analysis plan: `docs/phase3_analysis_plan.md`, committed before the runs):
* *Patient key* = the file name's leading C-number. It matches NIH's own patient-to-cell mapping for all 27,558 cells (200 patients; NIH lists patient C47 under two keys, one per microscope, which the C-number merges).
* *Patient split*: `StratifiedGroupKFold(5)` gives five candidate hold-out sets of ~20 %; we take the one closest to 20 % of the cells and 50 % parasitized (a rule on labels only). Then `StratifiedGroupKFold(10, random_state=50)` on the rest. The split is saved in `results/A1_slide_split/split.csv` so every machine trains on exactly the same one.
* *Statistics*: patients are the unit (all intervals resample whole patients); primary protocol = clean (un-augmented) evaluation, primary metric = MCC; paired contrasts use the Nadeau–Bengio corrected t-test over folds and a paired patient bootstrap of the ensemble; Holm correction over the three contrasts.

## 1 · Parameters

In [1]:
EXPERIMENTS = ["A1", "A2", "A3"]   # any of A1-A3 (A0 is the Phase 2 run; its folds are read for the comparison)
FOLDS_TO_TRAIN = []                # [k] on a pod, list(range(10)) on one machine, empty for tables only
AGGREGATE = True                   # build each experiment's tables once all 10 of its folds are in
DATA_DIR = "../outputs/data"       # NIH cell_images (downloaded if missing; git-ignored)
CACHE_DIR = "../outputs"           # decoded image caches, 4.1 GB each, one per preprocessing (git-ignored)
RESULTS_ROOT = "../results"        # results/<experiment folder>/: split.csv, folds/, tables, figures
FOLDS_ROOT = None                  # where new folds go (None: RESULTS_ROOT/<folder>/folds; pods: /workspace/out)
N_WORKERS = None                   # augmentation processes (None: the CPUs the container allows, minus 2)
XLA = True                         # XLA-compile the training step (speed only)
SMOKE = False                      # quick pipeline test: 5 % of the images, 2 epochs of 20 steps, live split
N_BOOT = 10_000                    # patient-bootstrap resamples for the hypothesis tests

In [2]:
# Parameters
EXPERIMENTS = ["A2"]
FOLDS_TO_TRAIN = [5]
AGGREGATE = False
DATA_DIR = "/workspace/data"
CACHE_DIR = "/workspace"
RESULTS_ROOT = "/workspace/repo/results"
FOLDS_ROOT = "/workspace/out"
N_WORKERS = None


In [3]:
%matplotlib inline
import json, os, sys, time
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "1")
os.environ.setdefault("TF_GPU_THREAD_MODE", "gpu_private")  # dedicated host threads launch the GPU kernels
import warnings; warnings.filterwarnings("ignore", category=DeprecationWarning)

SRC = next(p for p in ["../src", "src", "/workspace/repo/src"] if os.path.exists(os.path.join(p, "marques2022_exact.py")))
sys.path.insert(0, os.path.abspath(SRC))
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from sklearn.metrics import classification_report
import marques2022_exact as mx

TRAINING = len(FOLDS_TO_TRAIN) > 0
EXPECTED = json.load(open(os.path.join(RESULTS_ROOT, "ablation_splits.json")))  # committed split fingerprints
folder = lambda exp: os.path.join(RESULTS_ROOT, mx.EXPERIMENTS[exp]["folder"])
folds_dir = lambda exp: (os.path.join(FOLDS_ROOT, mx.EXPERIMENTS[exp]["folder"]) if FOLDS_ROOT
                         else os.path.join(folder(exp), "folds"))
display(pd.DataFrame(mx.EXPERIMENTS).T[["split", "preprocess", "question"]])
print("Recipe (A0's, from the paper's code):", json.dumps(mx.PAPER))
print("Experiments here:", EXPERIMENTS, "| folds to train:", FOLDS_TO_TRAIN or "none (tables only)")

,split,preprocess,question
A0,image,none,Baseline: the paper's protocol
A1,patient,none,H1: how much does patient leakage inflate the ...
A2,patient,yuv,H2: does YUV stain normalisation help on unsee...
A3,image,yuv,Control: does YUV change anything when patient...


Recipe (A0's, from the paper's code): {"SEED": 1234, "KFOLD": 10, "KFOLD_SEED": 50, "EPOCHS": 33, "BATCH_SIZE": 16, "IMG_SIZE": 224, "LR": 0.0001, "LR_FACTOR": 0.5, "PATIENCE": 6, "MIN_LR": 1e-06, "LABEL_SMOOTHING": 0.1, "VAL_FRACTION": 0.2, "WEIGHTS": "noisy-student", "DENSE": [128, 64, 32], "DROPOUT": 0.3}
Experiments here: ['A2'] | folds to train: [5]


## 2 · Data and the two splits
NIH malaria cell images: 27,558 cells from 200 patients (150 infected, 50 healthy). Each file name starts with the patient's C-number, e.g. `C33P1thinF_IMG_20150619_114756a_cell_179.png` → patient C33. Infected patients also contribute **uninfected** cells (10,364 of the 13,779), so under the paper's random image split almost every patient has cells on both sides.

* **Image split (A0, A3):** the paper's `train_test_split(test_size=0.2)` + `StratifiedKFold(10)`, rebuilt exactly as in A0 (its fingerprint is checked below).
* **Patient split (A1, A2):** the same proportions, but every patient's cells go to one side only. Patients differ a lot (65–702 cells; 1–90 % parasitized), so the folds' class mix varies more than under the image split; MCC and balanced accuracy are less sensitive to that than accuracy.

In [4]:
def describe(learn, val, folds):
    pts = lambda d: d["file"].map(mx.patient_id).nunique()
    share = lambda d: f"{100 * (d['label'] == 'Parasitized').mean():.1f} %"
    rows = [dict(part="learning set (10-fold CV)", cells=len(learn), patients=pts(learn), parasitized=share(learn)),
            dict(part="hold-out set", cells=len(val), patients=pts(val), parasitized=share(val))]
    rows += [dict(part=f"fold {k}: test part", cells=len(te), patients=pts(learn.iloc[te]), parasitized=share(learn.iloc[te]))
             for k, (_, te) in enumerate(folds)]
    return pd.DataFrame(rows).set_index("part")

if TRAINING:
    root = mx.find_or_download(DATA_DIR)
    df = mx.index_dataset(root)
    if SMOKE:  # like the paper's QUICK_AND_DIRTY switch; the committed split needs all images, so split live
        df = mx.build_index(df.sample(frac=0.05, random_state=mx.PAPER["SEED"])[["file", "label"]].values, root)
    print(f"Images: {len(df):,}", df["label"].value_counts().to_dict())
    splits = {}
    for exp in EXPERIMENTS:
        kind = mx.EXPERIMENTS[exp]["split"]
        if kind == "patient" and not SMOKE:
            splits[exp] = mx.load_split(os.path.join(folder(exp), "split.csv"), df)
        else:
            splits[exp] = mx.make_split(df, kind)
        if kind == "patient":
            mx.check_no_shared_patients(*splits[exp])
        fp = mx.split_fingerprint(*splits[exp])
        print(f"{exp}: {kind} split, fingerprint {fp}" + ("" if SMOKE else f" (committed: {EXPECTED[kind]})"))
        assert SMOKE or fp == EXPECTED[kind], f"{exp} would not train on the committed {kind} split"
    for kind in dict.fromkeys(mx.EXPERIMENTS[e]["split"] for e in EXPERIMENTS):
        exp = next(e for e in EXPERIMENTS if mx.EXPERIMENTS[e]["split"] == kind)
        print(f"\n{kind} split"); display(describe(*splits[exp]))
else:
    s = pd.read_csv(os.path.join(folder("A1"), "split.csv"))
    learn, val = s[s.role == "learn"].reset_index(drop=True), s[s.role == "holdout"].reset_index(drop=True)
    folds = [(np.flatnonzero(learn.fold != k), np.flatnonzero(learn.fold == k)) for k in range(learn.fold.max() + 1)]
    print("Patient split (A1, A2), from results/A1_slide_split/split.csv:"); display(describe(learn, val, folds))

Images: 27,558 {'Parasitized': 13779, 'Uninfected': 13779}
A2: patient split, fingerprint ff94f8c3b200e708 (committed: ff94f8c3b200e708)

patient split


,cells,patients,parasitized
part,,,
learning set (10-fold CV),22008,160,49.8 %
hold-out set,5550,40,50.7 %
fold 0: test part,3245,16,66.6 %
fold 1: test part,1661,16,29.8 %
fold 2: test part,2887,15,64.1 %
fold 3: test part,2194,16,49.3 %
fold 4: test part,1892,16,41.5 %
fold 5: test part,2005,17,42.4 %
fold 6: test part,3011,16,63.8 %


## 3 · The YUV preprocessing (H2)
Umer *et al.* (2020) convert each cell to YUV, equalise the brightness (Y) histogram, and convert back to RGB, so the network sees cells with the same brightness spread whatever the slide's stain and lighting. We do exactly that with OpenCV (`COLOR_BGR2YUV` → `equalizeHist` on Y → `COLOR_YUV2BGR`) at each image's native size, then apply A0's resize and augmentation unchanged. Colour (U, V) is left as it is. The pure-black background stays black (`equalizeHist` maps the lowest grey level to 0). Two deviations from Umer: no 120 × 120 resize (we keep 224, as A0) and no edge kernel (the paper does not specify it).

The equalisation is strong: a cell's narrow brightness range is stretched over 0–255, which also amplifies texture and noise (bottom row). Whether that helps or hurts on unseen patients is exactly what A2 vs A1 measures, and A3 vs A0 shows its effect when patients are shared.

In [5]:
fig_path = os.path.join(folder("A2"), "yuv_examples.png")
if os.path.exists(fig_path):
    display(Image(fig_path))

## 4 · Train the folds
Exactly A0's training loop (`mx.train_fold`), now with `experiment=` set: 33 epochs × ~1,200 steps of 16 images, validation on the fold's test part after every epoch, ReduceLROnPlateau, the best-val_loss checkpoint reloaded, then predictions on the test part and on the hold-out set, both clean and augmented. Each preprocessing gets its own image cache (`cell_images_224_none.npy`, `cell_images_224_yuv.npy`); a small `.json` next to each records how it was built, and `train_fold` refuses a cache that doesn't match the experiment.

On RunPod, each pod trains fold *k* of A1, A2 and A3 one after another (`src/runpod_10fold.py --experiments A1,A2,A3`), so the 10 pods finish all 30 models in about an hour.

In [6]:
def print_like_the_paper(res, exp):
    npz = np.load(os.path.join(folds_dir(exp), f"fold_{res['fold']}", f"fold_{res['fold']}_predictions.npz"))
    for name, title in [("test_clean", "Predicting... (CV test part, clean)"), ("val_clean", "Validating... (hold-out set, clean)"),
                        ("val_aug", "Validating... (hold-out set, augmented = paper protocol)")]:
        y_true = npz["test_true" if name.startswith("test") else "val_true"]
        pred = res["probs"][name].argmax(1)
        print(title, f"\nAccuracy {np.mean(pred == y_true):.6f}")
        print(classification_report(y_true, pred, labels=[0, 1], target_names=mx.TIPOS, digits=6, zero_division=0))

if TRAINING:
    workers = N_WORKERS or max(1, mx.available_cpus() - 2)
    smoke = dict(epochs=2, max_steps_per_epoch=20) if SMOKE else {}
    for exp in EXPERIMENTS:
        prep = mx.EXPERIMENTS[exp]["preprocess"]
        cache = os.path.join(CACHE_DIR, f"cell_images_224_{prep}{'_smoke' if SMOKE else ''}.npy")
        if not os.path.exists(cache + ".json"):  # the .json is written last, so a half-built cache is rebuilt
            t0 = time.time()
            mx.build_image_cache(list(df["path"]), cache, workers=mx.available_cpus(), preprocess=prep)
            print(f"Image cache ({prep}) built in {time.time() - t0:.0f} s")
        learn, val, folds = splits[exp]
        with mx.AugmentPool(workers, cache) as pool:
            for k in FOLDS_TO_TRAIN:
                print("*" * 50, f"\n** {exp} · {mx.EXPERIMENTS[exp]['split']} split · preprocess={prep} · fold {k}"
                      f"   ({workers} augmentation workers)")
                print(f"Found {len(folds[k][0])} / {len(folds[k][1])} / {len(val)} images (train / test / hold-out)")
                t0 = time.time()
                res = mx.train_fold(k, learn=learn, val=val, folds=folds, pool=pool, out_dir=folds_dir(exp), xla=XLA,
                                    experiment=exp, **smoke)
                print_like_the_paper(res, exp)
                print(f"Best epoch {res['best_epoch']} of {res['epochs']} · training {res['train_seconds'] / 60:.1f} min "
                      f"(median epoch {np.median(res['epoch_seconds']):.0f} s) · End of {exp} fold {k} - Elapsed Time: "
                      f"{time.strftime('%H:%M:%S', time.gmtime(time.time() - t0))}")

Image cache (yuv) built in 2 s


************************************************** 
** A2 · patient split · preprocess=yuv · fold 5   (11 augmentation workers)
Found 20003 / 2005 / 5550 images (train / test / hold-out)


2026-11-01 22:45:10.580923: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-11-01 22:45:10.580977: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-11-01 22:45:10.582082: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


2026-11-01 22:45:11.350439: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


Epoch 1/33


2026-11-01 22:45:19.414896: W tensorflow/compiler/tf2xla/kernels/random_ops.cc:59] Warning: Using tf.random.uniform with XLA compilation will ignore seeds; consider using tf.random.stateless_uniform instead if reproducible behavior is desired. efficientnetb0_marques2022/efficientnet-b0/block2b_drop/dropout/random_uniform/RandomUniform


... (63 outputs left out)

1250/1250 - 19s - loss: 0.2296 - accuracy: 0.9874 - val_loss: 0.2603 - val_accuracy: 0.9646 - lr: 1.2500e-05 - epoch_seconds: 19.4571 - 19s/epoch - 16ms/step


Epoch 33/33


1250/1250 - 20s - loss: 0.2301 - accuracy: 0.9880 - val_loss: 0.2562 - val_accuracy: 0.9701 - lr: 1.2500e-05 - epoch_seconds: 19.5420 - 20s/epoch - 16ms/step


Predicting... (CV test part, clean) 
Accuracy 0.969077
              precision    recall  f1-score   support

 Parasitized   0.965762  0.961222  0.963486       851
  Uninfected   0.971503  0.974870  0.973183      1154

    accuracy                       0.969077      2005
   macro avg   0.968632  0.968046  0.968335      2005
weighted avg   0.969066  0.969077  0.969068      2005

Validating... (hold-out set, clean) 
Accuracy 0.976216
              precision    recall  f1-score   support

 Parasitized   0.972193  0.981172  0.976662      2815
  Uninfected   0.980436  0.971115  0.975753      2735

    accuracy                       0.976216      5550
   macro avg   0.976314  0.976144  0.976208      5550
weighted avg   0.976255  0.976216  0.976214      5550

Validating... (hold-out set, augmented = paper protocol) 
Accuracy 0.973694
              precision    recall  f1-score   support

 Parasitized   0.971055  0.977265  0.974150      2815
  Uninfected   0.976445  0.970018  0.973221      27

## 5 · Results of each experiment
Built from the saved outputs of all 10 folds of each experiment (`mx.aggregate` re-checks that no patient is on both sides of any fold or of the hold-out split). Tables use **clean** evaluation; the ensemble is also reported under the paper's augmented protocol. Files per experiment: `results/<folder>/<ID>_summary.json`, per-fold CSVs, and `<ID>_curves_cm_roc.png`.

In [7]:
done = {exp: [k for k in range(10) if os.path.exists(os.path.join(folds_dir(exp), f"fold_{k}", f"fold_{k}.json"))]
        for exp in EXPERIMENTS}
for exp in EXPERIMENTS:
    print(f"{exp}: folds available {done[exp]}" + ("" if len(done[exp]) == 10 else "  → its tables need all 10 folds"))
READY = [exp for exp in EXPERIMENTS if AGGREGATE and len(done[exp]) == 10 and not SMOKE]
aggs = {}
for exp in READY:
    agg = mx.aggregate(folds_dir(exp))
    aggs[exp] = (agg, mx.write_results(agg, folder(exp)))

A2: folds available [5]  → its tables need all 10 folds


In [8]:
if aggs:
    import ablation_figures as figs  # only needed for the tables, not for training on a pod
for exp, (agg, summary) in aggs.items():
    display(Markdown(f"### {exp} · {mx.EXPERIMENTS[exp]['question']}"))
    pf = agg["per_fold"]["clean"].set_index("fold")
    cols = {"best_epoch": "best epoch", "minutes": "minutes", "cv_accuracy": "accuracy", "cv_recall": "recall",
            "cv_specificity": "specificity", "cv_mcc": "MCC", "cv_auc": "ROC-AUC", "holdout_accuracy": "hold-out acc.",
            "holdout_mcc": "hold-out MCC"}
    t = pf[list(cols)].rename(columns=cols)
    t = pd.concat([t, t.mean().to_frame("mean").T, t.std(ddof=1).to_frame("SD").T])
    display(t.round(2))
    for proto in ["clean", "paper"]:
        pred = agg["ens_probs"][proto].argmax(1)
        print(f"{exp} · 10-model ensemble on the {len(pred):,}-cell hold-out set ({summary['n_patients_holdout']} patients),"
              f" {proto} protocol · accuracy {100 * np.mean(pred == agg['val_true']):.2f} %")
        print(classification_report(agg["val_true"], pred, target_names=mx.TIPOS, digits=4))
    display(Image(figs.experiment_panels(agg, exp, os.path.join(folder(exp), f"{mx.EXPERIMENTS[exp]['prefix']}_curves_cm_roc.png"))))

## 6 · Hypothesis tests: A0 vs A1 vs A2 vs A3
`src/ablation_stats.py`, run on the saved predictions of all four experiments (A0's from Phase 2):
* **Paired contrasts** (A2 vs A1, A3 vs A0: the same cells in both arms): Nadeau–Bengio corrected resampled t-test on the 10 per-fold differences (the folds' training sets overlap by ~89 %, so a plain paired t-test would be too optimistic), and a paired patient bootstrap (resampling whole patients) of the ensemble's hold-out difference.
* **A1 vs A0** (different hold-out sets): independent patient bootstraps per arm; the fold-level Welch test is descriptive only.
* **Holm** correction over the three contrasts, per metric. Next to the tests: the deck's pre-registered rules.

Written to `results/ablation/` (clean protocol, primary) and `results/ablation/paper_protocol/`.

In [9]:
if not TRAINING and not SMOKE:
    import ablation_figures as figs
    import ablation_stats as st
    out = st.analyse(RESULTS_ROOT, n_boot=N_BOOT)
    st.analyse(RESULTS_ROOT, out_dir=os.path.join(RESULTS_ROOT, "ablation", "paper_protocol"), n_boot=N_BOOT, protocol="paper")
    display(Markdown(out["text"]))
    abl = os.path.join(RESULTS_ROOT, "ablation")
    if len(out["found"]) > 1:
        display(Image(figs.fold_strip(out["found"], "mcc", path=os.path.join(abl, "ablation_fold_mcc.png"))))
        display(Image(figs.fold_strip(out["found"], "accuracy", path=os.path.join(abl, "ablation_fold_accuracy.png"))))
    if out["contrasts"]:
        display(Image(figs.contrast_forest(out["contrasts"], path=os.path.join(abl, "ablation_contrasts.png"))))
        display(Image(figs.contrast_forest(out["contrasts"], level="fold", path=os.path.join(abl, "ablation_contrasts_folds.png"))))
    if {"A1", "A2"} <= set(out["found"]):
        display(Image(figs.per_patient(out["found"], "A1", "A2", path=os.path.join(abl, "ablation_per_patient.png"))))

## 7 · Limitations (stated before the results)
* **Cells, not patients.** The task is still single-cell classification; three quarters of the uninfected cells come from infected patients' smears. A patient-level diagnosis would need slide-level aggregation and is not tested here.
* **One hold-out split per hypothesis.** The patient hold-out has 40 patients; which 40 matters. The 10-fold CV spread and the patient bootstrap show how much that uncertainty is, but a different hold-out draw would move the numbers.
* **Labels from one expert.** Fuhad *et al.* (2020) questioned about 5 % of the cells; all results use the original labels, as the paper did.
* **Same hospital, shared imaging days.** A patient split removes patient leakage, not site or day effects; there is no external test set, so H2 is tested on unseen patients of the same lab, not on another lab's stain.
* **Run-to-run noise.** Each arm is one training run per fold; the corrected t-test uses the spread across folds, which includes training noise, but the ensemble bootstrap does not.

## 8 · Conclusion
*Written once the runs are in; see `results/ablation/ablation_summary.md`.*